# Insect Detection - Quick Demo (Pre-trained Models)
Run evaluation without training - uses pre-trained models if available

In [ ]:
# Import libraries
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.models import load_model
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

print(f"TensorFlow Version: {tf.__version__}")
print(f"GPU Available: {len(tf.config.list_physical_devices('GPU'))}")

In [ ]:
# Configuration
TEST_DIR = './AGRO/test'
BATCH_SIZE = 32
IMG_SIZE = 224

print(f"Test directory: {TEST_DIR}")
print(f"Image size: {IMG_SIZE}x{IMG_SIZE}")
print(f"Batch size: {BATCH_SIZE}")

In [ ]:
# Load test data
test_datagen = ImageDataGenerator(rescale=1./255)

test_generator = test_datagen.flow_from_directory(
    TEST_DIR,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    shuffle=False
)

print(f"Classes: {test_generator.class_indices}")
print(f"Total test samples: {test_generator.samples}")

In [ ]:
# Create class mapping
inv_map_classes = {v: k for k, v in test_generator.class_indices.items()}
print(f"Inverse class mapping: {inv_map_classes}")

In [ ]:
# Function to load models
def load_all_models():
    models = {}
    model_names = ['VGG_model.keras', 'ResNet50.keras', 'MobileNetV2.keras', 'Xception.keras']
    
    for model_name in model_names:
        try:
            model = load_model(model_name)
            models[model_name.replace('.keras', '')] = model
            print(f"✓ {model_name} loaded successfully")
        except Exception as e:
            print(f"✗ {model_name} not found: {e}")
    
    if not models:
        print("\n⚠ NO PRE-TRAINED MODELS FOUND!")
        print("Please ensure .keras files are in the current directory or")
        print("run the full notebook (NEW_FINAL_VERSION_8_9_INSECT_DETECTION.ipynb) to train models first.")
    
    return models

models = load_all_models()

In [ ]:
# Evaluate each model
if models:
    results = {}
    
    for model_name, model in models.items():
        print(f"\n{'='*50}")
        print(f"Evaluating {model_name}")
        print(f"{'='*50}")
        
        # Reset generator
        test_generator.reset()
        
        # Make predictions
        predictions = model.predict(test_generator)
        pred_classes = np.argmax(predictions, axis=1)
        true_classes = test_generator.classes
        
        # Calculate accuracy
        accuracy = accuracy_score(true_classes, pred_classes)
        print(f"Accuracy: {accuracy*100:.2f}%")
        
        results[model_name] = {
            'accuracy': accuracy,
            'predictions': pred_classes,
            'true_classes': true_classes,
            'model': model
        }
else:
    print("Cannot proceed - no models loaded")

In [ ]:
# Show confusion matrix for first model
if models:
    first_model = list(results.keys())[0]
    pred = results[first_model]['predictions']
    true = results[first_model]['true_classes']
    
    cm = confusion_matrix(true, pred)
    
    plt.figure(figsize=(12, 10))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
                xticklabels=[inv_map_classes[i] for i in range(len(inv_map_classes))],
                yticklabels=[inv_map_classes[i] for i in range(len(inv_map_classes))])
    plt.title(f'Confusion Matrix - {first_model}')
    plt.ylabel('True Label')
    plt.xlabel('Predicted Label')
    plt.tight_layout()
    plt.savefig(f'confusion_matrix_{first_model}.png', dpi=100, bbox_inches='tight')
    plt.show()
    print(f"Confusion matrix saved")

In [ ]:
# Compare model accuracies
if models:
    accuracies = {model: results[model]['accuracy']*100 for model in results}
    
    plt.figure(figsize=(10, 6))
    plt.bar(accuracies.keys(), accuracies.values(), color=['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728'][:len(accuracies)])
    plt.ylabel('Accuracy (%)')
    plt.title('Model Accuracy Comparison')
    plt.ylim([0, 105])
    for k, v in accuracies.items():
        plt.text(k, v+1, f'{v:.2f}%', ha='center')
    plt.tight_layout()
    plt.savefig('model_accuracy_comparison.png', dpi=100, bbox_inches='tight')
    plt.show()
    print(f"Accuracy comparison saved")

In [ ]:
# Classification report for first model
if models:
    first_model = list(results.keys())[0]
    pred = results[first_model]['predictions']
    true = results[first_model]['true_classes']
    
    class_names = [inv_map_classes[i] for i in range(len(inv_map_classes))]
    report = classification_report(true, pred, target_names=class_names)
    print(f"\nDetailed Classification Report - {first_model}:")
    print(report)

## Summary

✅ If you see model accuracies above, the pre-trained models are working!  
❌ If you see "NO PRE-TRAINED MODELS FOUND", run the full notebook first to train and save models.